# Lab: Neural Networks From Scratch, PyTorch, and Optimizers

**Purpose:** introduce core concepts and practice deep learning coding

You will implement a two-hidden-layer neural network for handwritten digit classification. The dataset is the built-in `sklearn` digits dataset: 8x8 grayscale images, labeled 0 through 9.

## Big Picture

A neural network training pipeline has the same structure in many data mining applications:

`raw data -> preprocessing -> model -> loss -> gradients -> optimizer update -> evaluation`

In this lab:

- **Raw data:** 8x8 handwritten digit images.
- **Preprocessing:** standardize pixel features using training-set statistics.
- **Model:** a two-hidden-layer multilayer perceptron, also called an MLP.
- **Loss:** multiclass cross-entropy.
- **Gradients:** first written manually in NumPy, then handled by PyTorch autograd.
- **Optimizer:** full-gradient descent, mini-batch SGD, RMSProp, and Adam.

Keep this pipeline in mind. The details change across projects, but this overall flow is very common.


## Learning Goals

After completing this lab, you should be able to:

- Preprocess data without leaking validation/test information into training.
- Implement a two-hidden-layer neural network with NumPy.
- Implement softmax cross-entropy and backpropagation for multiclass classification.
- Compare full-gradient descent, mini-batch SGD, RMSProp, and Adam.
- Use PyTorch tensors, `nn.Module`, `DataLoader`, and `torch.optim`.
- Draw and interpret loss curves.


## 0. Setup

Run this cell first. You should not need to edit it.


In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

try:
    import torch
    from torch import nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    print("PyTorch is not available:", exc)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## 1. Dataset: Handwritten Digits

The input `X_raw` has shape `(n_samples, 64)`. Each row is one flattened 8x8 image. The label `y_raw` is an integer from 0 to 9.


### What Are the Features and Labels?

Each image is 8 by 8 pixels. We flatten it into a vector of length 64:

`image shape (8, 8) -> feature vector shape (64,)`

The label is one integer:

`0, 1, 2, ..., 9`

The model will output 10 scores, one score for each digit class. The predicted class is the index with the largest score.

Example shape convention used in this lab:

- `X`: shape `(number_of_examples, 64)`
- `y`: shape `(number_of_examples,)`
- model output logits: shape `(number_of_examples, 10)`
- predicted labels: shape `(number_of_examples,)`


In [ ]:
digits = load_digits()
X_raw = digits.data.astype(np.float64)       # shape: (n_samples, 64)
y_raw = digits.target.astype(np.int64)       # labels: 0, 1, ..., 9
images = digits.images                       # shape: (n_samples, 8, 8)
class_names = [str(i) for i in range(10)]

print("X shape:", X_raw.shape)
print("y shape:", y_raw.shape)
print("Classes:", np.unique(y_raw))

fig, axes = plt.subplots(2, 5, figsize=(8, 3.2))
for ax, image, label in zip(axes.ravel(), images[:10], y_raw[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"label={label}")
    ax.axis("off")
plt.suptitle("Example 8x8 handwritten digit images")
plt.tight_layout()
plt.show()


## 2. TODO: Split and Preprocess the Data

Complete the next cell.

Requirements:

- Create train, validation, and test splits.
- Fit standardization on the training split only.
- Apply that same transformation to all splits.


### Why Split the Data?

We use three splits:

- **Training set:** used to update model parameters.
- **Validation set:** used to compare choices such as optimizer, learning rate, and hidden size.
- **Test set:** used only at the end to estimate performance on unseen data.

Do not tune your model based on the test set. If you repeatedly use the test set to make decisions, it stops being a fair final evaluation.


### Preprocessing Hints

For `train_val_test_split`:

- Let `n = len(X)`.
- Create shuffled indices with `rng.permutation(n)`.
- Compute `n_train = int(train_frac * n)`.
- Compute `n_val = int(val_frac * n)`.
- Slice the shuffled indices into train, validation, and test indices.

For `standardize_train_val_test`:

- Compute `mean = X_train_raw.mean(axis=0, keepdims=True)`.
- Compute `std = X_train_raw.std(axis=0, keepdims=True) + 1e-8`.
- Use the same `mean` and `std` for all splits.

Important: some digit pixels are always zero in the training set, so their standardized standard deviation can remain zero. That is okay.


In [ ]:
def train_val_test_split(X, y, train_frac=0.70, val_frac=0.15, seed=42):
    """Return X_train, y_train, X_val, y_val, X_test, y_test."""
    raise NotImplementedError("TODO: implement train_val_test_split")


def standardize_train_val_test(X_train_raw, X_val_raw, X_test_raw):
    """Fit standardization on train only, then transform train/val/test."""
    raise NotImplementedError("TODO: implement standardize_train_val_test")


X_train_raw, y_train, X_val_raw, y_val, X_test_raw, y_test = train_val_test_split(X_raw, y_raw)
X_train, X_val, X_test, feature_mean, feature_std = standardize_train_val_test(
    X_train_raw, X_val_raw, X_test_raw
)

print("Train / val / test sizes:", len(X_train), len(X_val), len(X_test))
print("Training feature mean range after standardization:", X_train.mean(axis=0).min().round(5), X_train.mean(axis=0).max().round(5))
print("Training feature std range after standardization:", X_train.std(axis=0).min().round(5), X_train.std(axis=0).max().round(5))


### Check Your Preprocessing


In [ ]:
assert X_train.shape[1] == 64
assert y_train.ndim == 1
assert len(X_train) + len(X_val) + len(X_test) == len(X_raw)
assert np.allclose(X_train.mean(axis=0), np.zeros(64), atol=1e-6)
print("Preprocessing checks passed.")


## 3. TODO: NumPy Neural Network

Implement this two-hidden-layer MLP:

`64 -> Linear(h1) -> ReLU -> Linear(h2) -> ReLU -> Linear(10) -> Softmax`

You need multiclass cross-entropy loss:

`loss = mean(-log(probability assigned to the correct class))`


### Model Shapes

Use this architecture:

`64 -> hidden_dim1 -> hidden_dim2 -> 10`

For default hidden sizes `hidden_dim1=64` and `hidden_dim2=32`, parameter shapes should be:

- `W1`: `(64, 64)` and `b1`: `(1, 64)`
- `W2`: `(64, 32)` and `b2`: `(1, 32)`
- `W3`: `(32, 10)` and `b3`: `(1, 10)`

If `X_batch` has shape `(batch_size, 64)`, then:

- first hidden activations have shape `(batch_size, hidden_dim1)`
- second hidden activations have shape `(batch_size, hidden_dim2)`
- logits have shape `(batch_size, 10)`
- probabilities have shape `(batch_size, 10)`


### Forward Pass Plan

Implement `forward_numpy` in this order:

1. `Z1 = X @ W1 + b1`
2. `A1 = relu(Z1)`
3. `Z2 = A1 @ W2 + b2`
4. `A2 = relu(Z2)`
5. `logits = A2 @ W3 + b3`
6. `probs = softmax(logits)`
7. Return `probs` and a `cache` dictionary containing the intermediate values needed for backprop.

The cache should include at least `X`, `Z1`, `A1`, `Z2`, `A2`, `logits`, and `probs`.


### Loss and Softmax Hints

Softmax turns logits into probabilities. For numerical stability, subtract the largest logit in each row before exponentiating:

`shifted = logits - max(logits in each row)`

Then:

`exp_scores = exp(shifted)`

`probs = exp_scores / row_sum(exp_scores)`

For cross-entropy, select the probability assigned to the correct class for each example. If `y[i]` is the correct label for example `i`, use:

`probs[i, y[i]]`

The loss is the average negative log of those probabilities.


### Backpropagation Guide

For softmax plus cross-entropy, the gradient with respect to logits has a convenient form:

1. Start with `dlogits = probs.copy()`.
2. For each example `i`, subtract `1` from `dlogits[i, y[i]]`.
3. Divide by the batch size.

Then backpropagate through the layers in reverse order:

`logits -> W3,b3 -> ReLU2 -> W2,b2 -> ReLU1 -> W1,b1`

Useful matrix-gradient patterns:

- If `Y = A @ W + b`, then `dW = A.T @ dY`.
- `db = sum(dY over rows, keepdims=True)`.
- `dA = dY @ W.T`.
- ReLU gradient is zero where the pre-activation `Z <= 0` and unchanged where `Z > 0`.


In [ ]:
def init_params(input_dim=64, hidden_dim1=64, hidden_dim2=32, num_classes=10, seed=42):
    raise NotImplementedError("TODO: initialize W1, b1, W2, b2, W3, b3")


def relu(z):
    raise NotImplementedError("TODO: implement ReLU")


def softmax(logits):
    raise NotImplementedError("TODO: implement stable softmax")


def forward_numpy(X, params):
    """Return class probabilities and a cache for backprop."""
    raise NotImplementedError("TODO: implement forward pass")


def cross_entropy_loss(probs, y):
    raise NotImplementedError("TODO: implement multiclass cross-entropy")


def backward_numpy(y, params, cache):
    """Return gradients matching every parameter shape."""
    raise NotImplementedError("TODO: implement backpropagation")


def predict_numpy(X, params):
    raise NotImplementedError("TODO: return predicted class ids")


def accuracy_numpy(X, y, params):
    raise NotImplementedError("TODO: return accuracy as a float")


def evaluate_loss_numpy(X, y, params):
    probs, _ = forward_numpy(X, params)
    return cross_entropy_loss(probs, y)


### Check Your NumPy Network Shapes


In [ ]:
params = init_params(input_dim=X_train.shape[1], hidden_dim1=32, hidden_dim2=16, num_classes=10, seed=1)
probs, cache = forward_numpy(X_train[:7], params)
grads = backward_numpy(y_train[:7], params, cache)

assert probs.shape == (7, 10)
assert np.allclose(probs.sum(axis=1), 1.0)
for name in params:
    assert grads[name].shape == params[name].shape, (name, grads[name].shape, params[name].shape)
print("Shape checks passed.")


## 4. TODO: NumPy Optimizers and Training Loop

Implement full-gradient descent, mini-batch SGD, and RMSProp.


### Training Loop Pseudocode

A typical training loop looks like this:

```text
initialize parameters
for each epoch:
    create batches
    for each batch:
        probs, cache = forward pass
        loss = cross entropy
        grads = backward pass
        update parameters with optimizer
    compute train loss and validation loss
    compute train accuracy and validation accuracy
```

Record metrics once per epoch so that you can plot learning curves.


### Optimizer Hints

For full-gradient descent and mini-batch SGD, the update rule is the same:

`parameter = parameter - learning_rate * gradient`

The difference is the amount of data used to compute each gradient:

- Full-gradient descent uses the whole training set in one batch.
- Mini-batch SGD uses smaller batches, such as 64 examples.

For RMSProp, keep a running average of squared gradients:

`cache = rho * cache + (1 - rho) * gradient^2`

Then scale the update:

`parameter = parameter - learning_rate * gradient / (sqrt(cache) + epsilon)`

RMSProp often allows more stable training when different parameters have gradients at different scales.


### What Results Should Look Like?

Exact numbers vary, but after a correct implementation you should usually see:

- training loss decreases over epochs
- validation loss decreases at first, then may flatten
- validation accuracy substantially above random guessing
- random guessing for 10 classes is about 10% accuracy
- a reasonable implementation should often reach above 90% validation accuracy on this dataset

If accuracy stays near 10%, check softmax, cross-entropy, and `dlogits` first.


In [ ]:
def make_batches(X, y, batch_size, rng, shuffle=True):
    """Yield full data once when batch_size is None; otherwise yield mini-batches."""
    raise NotImplementedError("TODO: implement batching")


def update_params(params, grads, optimizer, lr, rms_cache=None, rmsprop_rho=0.9, eps=1e-8):
    """Apply one optimizer update in-place and return rms_cache."""
    raise NotImplementedError("TODO: implement parameter updates")


def train_numpy_model(
    optimizer="full_gd",
    hidden_dim1=64,
    hidden_dim2=32,
    lr=0.05,
    epochs=120,
    batch_size=64,
    seed=42,
):
    rng = np.random.default_rng(seed)
    params = init_params(
        input_dim=X_train.shape[1],
        hidden_dim1=hidden_dim1,
        hidden_dim2=hidden_dim2,
        num_classes=10,
        seed=seed,
    )
    rms_cache = {k: np.zeros_like(v) for k, v in params.items()}
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        raise NotImplementedError("TODO: implement training loop")

    return params, history


### Run NumPy Optimizer Experiments


In [ ]:
numpy_experiments = {
    "Full gradient descent": dict(optimizer="full_gd", lr=0.18, epochs=220, batch_size=None),
    "Mini-batch SGD": dict(optimizer="sgd", lr=0.05, epochs=120, batch_size=64),
    "RMSProp": dict(optimizer="rmsprop", lr=0.0015, epochs=120, batch_size=64),
}

numpy_results = {}
for name, cfg in numpy_experiments.items():
    params, history = train_numpy_model(**cfg, hidden_dim1=64, hidden_dim2=32, seed=7)
    numpy_results[name] = {"params": params, "history": history}
    print(
        f"{name:22s} | "
        f"final val loss={history['val_loss'][-1]:.4f} | "
        f"val acc={history['val_acc'][-1]:.3f} | "
        f"test acc={accuracy_numpy(X_test, y_test, params):.3f}"
    )


### Plot Loss Curves


In [ ]:
def plot_loss_curves(results, title):
    plt.figure(figsize=(9, 5))
    for name, result in results.items():
        plt.plot(result["history"]["train_loss"], label=f"{name} train", alpha=0.75)
        plt.plot(result["history"]["val_loss"], label=f"{name} val", linestyle="--")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title)
    plt.legend()
    plt.show()

plot_loss_curves(numpy_results, "NumPy two-hidden-layer network: optimizer comparison")


## 5. PyTorch Intro: Autograd


### NumPy vs PyTorch

In the NumPy section, you manually implement:

- forward pass
- loss
- backward pass
- optimizer updates

In PyTorch, you still write the model and training loop, but PyTorch handles the gradients:

- `loss.backward()` computes gradients.
- `optimizer.step()` updates parameters.
- `optimizer.zero_grad()` clears old gradients before the next update.

This is why understanding the NumPy version helps: PyTorch automates the same math, but you still need to know what is happening.


In [ ]:
if TORCH_AVAILABLE:
    x = torch.tensor([2.0], requires_grad=True)
    z = x**2 + 3*x + 1
    z.backward()
    print("z = x^2 + 3x + 1 at x=2:", z.item())
    print("dz/dx at x=2 should be 7:", x.grad.item())
else:
    print("Skipping PyTorch cells because PyTorch is not available.")


## 6. TODO: PyTorch Data Preparation

Convert NumPy arrays to PyTorch tensors and create `TensorDataset` objects.

For `CrossEntropyLoss`, labels must be integer class ids with dtype `torch.long`.


### PyTorch Data Hints

Use these dtypes:

- `X_train_t = torch.tensor(X_train, dtype=torch.float32)`
- `y_train_t = torch.tensor(y_train, dtype=torch.long)`

Why `torch.long` for labels? `nn.CrossEntropyLoss` expects class ids, not one-hot vectors.

Create datasets like:

`train_ds = TensorDataset(X_train_t, y_train_t)`

A `DataLoader` then handles batching and shuffling.


In [ ]:
if TORCH_AVAILABLE:
    X_train_t = None
    y_train_t = None
    X_val_t = None
    y_val_t = None
    X_test_t = None
    y_test_t = None

    train_ds = None
    val_ds = None
    test_ds = None

    xb, yb = next(iter(DataLoader(train_ds, batch_size=4, shuffle=True)))
    print("One mini-batch X shape:", xb.shape)
    print("One mini-batch y shape:", yb.shape)
else:
    print("Skipping PyTorch data setup.")


## 7. TODO: PyTorch Two-Hidden-Layer MLP

The model should output logits with shape `(batch_size, 10)`.


### PyTorch Model Hints

Your PyTorch model should match the NumPy architecture:

`64 -> hidden_dim1 -> hidden_dim2 -> 10`

A convenient implementation uses `nn.Sequential`:

```python
self.net = nn.Sequential(
    nn.Linear(input_dim, hidden_dim1),
    nn.ReLU(),
    nn.Linear(hidden_dim1, hidden_dim2),
    nn.ReLU(),
    nn.Linear(hidden_dim2, num_classes),
)
```

The final layer should not include `Softmax`. `nn.CrossEntropyLoss` expects raw logits and applies the needed log-softmax internally.


In [ ]:
if TORCH_AVAILABLE:
    class TorchMLP(nn.Module):
        def __init__(self, input_dim=64, hidden_dim1=64, hidden_dim2=32, num_classes=10):
            super().__init__()
            raise NotImplementedError("TODO: define layers")

        def forward(self, x):
            raise NotImplementedError("TODO: return logits")

    def torch_accuracy(model, X, y):
        raise NotImplementedError("TODO: compute multiclass accuracy")

    model = TorchMLP()
    logits = model(X_train_t[:5])
    print("Logits shape:", logits.shape)
else:
    print("Skipping model definition.")


## 8. TODO: PyTorch Training Loop and Optimizers


### PyTorch Training Loop Pseudocode

```text
for each epoch:
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    compute train/validation loss and accuracy without gradients
```

Common mistakes:

- Forgetting `optimizer.zero_grad()`.
- Applying softmax before `CrossEntropyLoss`.
- Using float labels instead of integer `torch.long` labels.
- Measuring validation metrics while gradients are still being tracked.


### Optimizer Selection Hints

Use these PyTorch optimizers:

- `torch.optim.SGD(model.parameters(), lr=lr)` for full-gradient descent and mini-batch SGD.
- `torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.9)` for RMSProp.
- `torch.optim.Adam(model.parameters(), lr=lr)` for Adam.

For full-gradient descent, use a DataLoader with one batch containing the full training set:

`DataLoader(train_ds, batch_size=len(train_ds), shuffle=True)`

For mini-batch methods, use `batch_size=64`.


In [ ]:
if TORCH_AVAILABLE:
    def eval_torch_loss(model, X, y, loss_fn):
        raise NotImplementedError("TODO: evaluate loss without gradients")

    def train_torch_model(optimizer_name="sgd", lr=0.05, epochs=80, batch_size=64, hidden_dim1=64, hidden_dim2=32, seed=42):
        torch.manual_seed(seed)
        model = TorchMLP(input_dim=X_train_t.shape[1], hidden_dim1=hidden_dim1, hidden_dim2=hidden_dim2, num_classes=10)
        loss_fn = nn.CrossEntropyLoss()

        loader = None
        optimizer = None

        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        for epoch in range(epochs):
            raise NotImplementedError("TODO: train for one epoch and record metrics")
        return model, history
else:
    print("Skipping PyTorch training helpers.")


### Run PyTorch Optimizer Experiments


In [ ]:
if TORCH_AVAILABLE:
    torch_experiments = {
        "Full gradient descent": dict(optimizer_name="full_gd", lr=0.12, epochs=140, batch_size=None),
        "Mini-batch SGD": dict(optimizer_name="sgd", lr=0.05, epochs=80, batch_size=64),
        "RMSProp": dict(optimizer_name="rmsprop", lr=0.001, epochs=80, batch_size=64),
        "Adam": dict(optimizer_name="adam", lr=0.003, epochs=80, batch_size=64),
    }

    torch_results = {}
    for name, cfg in torch_experiments.items():
        model, history = train_torch_model(**cfg, hidden_dim1=64, hidden_dim2=32, seed=7)
        torch_results[name] = {"model": model, "history": history}
        print(
            f"{name:22s} | "
            f"final val loss={history['val_loss'][-1]:.4f} | "
            f"val acc={history['val_acc'][-1]:.3f} | "
            f"test acc={torch_accuracy(model, X_test_t, y_test_t):.3f}"
        )
else:
    torch_results = {}
    print("Skipping PyTorch optimizer comparison.")


In [ ]:
if TORCH_AVAILABLE:
    plot_loss_curves(torch_results, "PyTorch two-hidden-layer network: optimizer comparison")
else:
    print("No PyTorch results to plot.")


## 9. Reflection Questions

1. Why should preprocessing be fit only on the training split?
2. How did the move from binary classification to 10-class classification change the output layer and loss?
3. What did the second hidden layer add to the model?
4. Which optimizer reached a good validation loss fastest?
5. What did PyTorch automate compared with your NumPy implementation?


## 10. Optional Extensions

- Try hidden sizes such as `(32, 16)`, `(128, 64)`, or `(128, 128)`.
- Add dropout in the PyTorch model.
- Add L2 regularization to the NumPy update.
- Implement momentum SGD in NumPy.
- Plot the examples that the best model misclassified.
